# CASCADE — Tier 1.5 · SPO+ decision-focused fine-tune  (v2)

A short fine-tune so the model's severity ranking minimizes **decision regret** (Elmachtoub-Grigas
SPO+). v1 failed two ways; v2 fixes both:

1. **SPO+ sign for a MAXIMIZATION decision.** We pick the top-B incidents to *maximize* captured
   severity, so the subgradient is `2(w*(2c_hat - c) - w*(c))` (the negation of the textbook
   minimization form). v1 had it backwards and climbed regret.
2. **Dedicated severity head + multi-task anchor.** v1 defined severity as `E[duration]xP(closure)`,
   so SPO+ wrecked the calibrated PMF (IBS blew up). v2 adds a separate `sev` head for the decision
   and keeps the **full Tier-1 loss as an anchor** (with the saved uncertainty weights) so survival /
   point-process / closure stay calibrated.
3. Select the epoch with the **lowest validation regret**.

### Upload
`train_bundle.npz`, `train_bundle_meta.json`, **and `trunk_mtl.pt`**.

### Download back into `models/`
`trunk_spo.pt`, `preds_spo.npz` (decision-tuned `severity`; survival metrics should stay ~Tier-1).


In [ ]:
# 0 · setup
import subprocess, sys
def pip(*p): subprocess.run([sys.executable,'-m','pip','install','-q',*p], check=False)
pip('scikit-survival')
import os, json, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from sksurv.metrics import concordance_index_censored
device='cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print('torch',torch.__version__,'| device',device)

In [ ]:
# 1 · load bundle + Tier-1 weights
for f in ['train_bundle.npz','train_bundle_meta.json','trunk_mtl.pt']:
    if not os.path.exists(f):
        from google.colab import files; print('upload train_bundle.npz, train_bundle_meta.json, trunk_mtl.pt'); files.upload(); break
z=np.load('train_bundle.npz',allow_pickle=True); meta=json.load(open('train_bundle_meta.json'))
ckpt=torch.load('trunk_mtl.pt',map_location=device)
N,V,K,L=meta['n_rows'],meta['num_nodes'],meta['n_bins'],meta['hist_len']
print('loaded | N',N,'K',K,'| has log_sigma:', 'log_sigma' in ckpt)

In [ ]:
# 2 · tensors + adjacency
def T(n,dt): return torch.as_tensor(z[n],dtype=dt).to(device)
X_num=T('X_num',torch.float32); X_cat=T('X_cat',torch.long)
node_id=T('node_id',torch.long); hist_idx=T('hist_idx',torch.long)
node_feat=T('node_feat',torch.float32); edge_index=T('edge_index',torch.long)
bin_idx=T('bin_idx',torch.long); road=T('road_closure',torch.long); prio=T('priority_high',torch.long)
durations=z['durations'].astype('float32'); events=z['events'].astype('int64'); cuts=z['cuts'].astype('float32')
split=z['split']; ev_t=torch.as_tensor(events,dtype=torch.float32,device=device)
tte=z['tte_min'].astype('float32'); tte_obs=z['tte_observed'].astype('int64')
dt_h=torch.as_tensor(np.clip(tte/60.0,1e-3,720.0),dtype=torch.float32,device=device)
tte_obs_t=torch.as_tensor(tte_obs,dtype=torch.float32,device=device)
def norm_adj(ei,V):
    loops=torch.arange(V,device=ei.device); ei=torch.cat([ei,torch.stack([loops,loops])],1)
    src,dst=ei[0],ei[1]; deg=torch.zeros(V,device=ei.device).scatter_add_(0,dst,torch.ones_like(dst,dtype=torch.float32))
    di=deg.pow(-0.5); di[torch.isinf(di)]=0.0; return torch.sparse_coo_tensor(ei,di[src]*di[dst],(V,V)).coalesce()
A=norm_adj(edge_index,V)
tr_idx=torch.where(torch.as_tensor(split==0))[0].to(device); va_idx=torch.where(torch.as_tensor(split==1))[0].to(device); te_idx=torch.where(torch.as_tensor(split==2))[0].to(device)
va_t,va_e=durations[split==1],events[split==1]
CAP=1440.0; mids=np.minimum(0.5*(cuts[:-1]+cuts[1:]),CAP).astype('float32'); mids_t=torch.as_tensor(mids,device=device)
dur_t=torch.as_tensor(durations,device=device)
# realized decision cost per incident: capped duration x road-closure x observed (censored -> 0)
cost_t=torch.clamp(dur_t,max=CAP)*road.float()*ev_t

In [ ]:
# 3 · model (Tier-1 arch) + NEW dedicated severity head
class GCN(nn.Module):
    def __init__(s,fin,h,out): super().__init__(); s.l1=nn.Linear(fin,h); s.l2=nn.Linear(h,out)
    def forward(s,x,A): return torch.sparse.mm(A,s.l2(F.relu(torch.sparse.mm(A,s.l1(x)))))
class EventFeat(nn.Module):
    def __init__(s,n_num,cards,emb=8,out=64):
        super().__init__(); s.embs=nn.ModuleList([nn.Embedding(c,emb) for c in cards]); s.proj=nn.Linear(n_num+emb*len(cards),out)
    def forward(s,xn,xc): return s.proj(torch.cat([xn]+[e(xc[:,i]) for i,e in enumerate(s.embs)],1))
class Trunk(nn.Module):
    def __init__(s,n_num,cards,fnode,d=64,p=0.2):
        super().__init__(); s.d=d; s.feat=EventFeat(n_num,cards,out=d); s.gcn=GCN(fnode,64,d)
        s.gru=nn.GRU(d,d,batch_first=True); s.fuse=nn.Sequential(nn.Linear(d*3,d),nn.ReLU(),nn.Dropout(p),nn.Linear(d,d))
    def event_vectors(s,xn,xc): return s.feat(xn,xc)
    def forward(s,idx,av):
        e=av[idx]; g=s.gcn(node_feat,A)[node_id[idx]]; bank=torch.cat([av,torch.zeros(1,s.d,device=av.device)],0)
        _,hn=s.gru(bank[hist_idx[idx]]); return s.fuse(torch.cat([e,g,hn.squeeze(0)],1))
class MMoE(nn.Module):
    def __init__(s,d,n_experts=4,n_tasks=4):
        super().__init__(); s.experts=nn.ModuleList([nn.Sequential(nn.Linear(d,d),nn.ReLU()) for _ in range(n_experts)]); s.gates=nn.ModuleList([nn.Linear(d,n_experts) for _ in range(n_tasks)])
    def forward(s,H):
        E=torch.stack([ex(H) for ex in s.experts],1); return [(F.softmax(g(H),-1).unsqueeze(-1)*E).sum(1) for g in s.gates]
class Heads(nn.Module):
    def __init__(s,d,K):
        super().__init__(); s.dur=nn.Linear(d,K); s.pp=nn.Linear(d,1); s.clo=nn.Linear(d,1); s.pri=nn.Linear(d,1)
    def forward(s,reps):
        lam=F.softplus(s.pp(reps[1]).squeeze(-1))+1e-4; return s.dur(reps[0]),lam,s.clo(reps[2]).squeeze(-1),s.pri(reps[3]).squeeze(-1)
cards=meta['cat_cardinalities']; n_num=X_num.shape[1]; fnode=node_feat.shape[1]; d=ckpt['config']['d']
def build():
    return (Trunk(n_num,cards,fnode,d=d).to(device), MMoE(d).to(device), Heads(d,K).to(device))
def load_mtl(tr,mm,hd): tr.load_state_dict(ckpt['trunk']); mm.load_state_dict(ckpt['mmoe']); hd.load_state_dict(ckpt['heads'])
trunk,mmoe,heads=build(); load_mtl(trunk,mmoe,heads)
sev_head=nn.Linear(d,1).to(device)                         # NEW: residual decision-cost head
nn.init.zeros_(sev_head.weight); nn.init.zeros_(sev_head.bias)   # start as identity (correction=0)
log_sigma=ckpt['log_sigma'].to(device) if 'log_sigma' in ckpt else torch.zeros(4,device=device)  # frozen anchor weights
print('initialized SPO+ model (Tier-1 weights + fresh severity head)')

In [ ]:
# 4 · losses + CORRECT maximization SPO+
def deephit_nll(logp,bins,ev):
    pmf=logp.exp(); idx=bins.clamp(0,pmf.shape[1]-1)[:,None]; lp=logp.gather(1,idx).squeeze(1)
    tail=torch.flip(torch.cumsum(torch.flip(pmf,[1]),1),[1]).gather(1,idx).squeeze(1).clamp_min(1e-8).log()
    return -(ev*lp+(1-ev)*tail).mean()
def rank_loss(pmf,bins,ev,sig=0.1,npairs=4096):
    cif=torch.cumsum(pmf,1); dev=ev.bool()
    if dev.sum()<2: return pmf.sum()*0.0
    B=pmf.shape[0]; ii=torch.randint(0,B,(npairs,),device=pmf.device); jj=torch.randint(0,B,(npairs,),device=pmf.device)
    v=dev[ii]&(bins[ii]<bins[jj])
    if v.sum()==0: return pmf.sum()*0.0
    ii,jj=ii[v],jj[v]; ti=bins[ii].clamp(0,pmf.shape[1]-1); return torch.exp(-(cif[ii,ti]-cif[jj,ti])/sig).mean()
def pp_nll(lam,dt,obs): lam=lam.clamp(1e-6,1e3); return (-obs*torch.log(lam)+lam*dt).mean()

def forward_all(tr,mm,hd,sh,idx):
    av=tr.event_vectors(X_num,X_cat); reps=mm(tr(idx,av)); dlog,lam,clog,plog=hd(reps)
    base=(F.softmax(dlog,1)*mids_t).sum(1)*torch.sigmoid(clog)+1e-4   # accuracy heuristic E[dur]xP(closure)
    corr=sh(reps[0]).squeeze(-1).clamp(-2,2)                          # learned residual (starts at 0)
    sev=base*torch.exp(corr)                                         # warm-started at base
    return dlog,lam,clog,plog,sev

def anchor_loss(dlog,lam,clog,plog,idx):
    logp=F.log_softmax(dlog,1)
    Lsv=deephit_nll(logp,bin_idx[idx],ev_t[idx])+0.5*rank_loss(logp.exp(),bin_idx[idx],ev_t[idx])
    Lpp=pp_nll(lam,dt_h[idx],tte_obs_t[idx])
    Lclo=F.binary_cross_entropy_with_logits(clog,road[idx].float())
    pm=prio[idx]>=0
    Lpri=F.binary_cross_entropy_with_logits(plog[pm],prio[idx][pm].float()) if pm.any() else Lsv*0
    raw=[Lsv,Lpp,Lclo,Lpri]
    return sum(0.5*torch.exp(-log_sigma[k])*raw[k]+0.5*log_sigma[k] for k in range(4))

def topk(cost,frac=0.1):
    B=max(1,int(frac*cost.numel())); w=torch.zeros_like(cost); w[torch.topk(cost,B).indices]=1.0; return w
def spo_plus_max(sev,cost,frac=0.1):
    # MAXIMIZATION SPO+: subgradient wrt sev = 2(w*(2sev-c) - w*(c)); surrogate has exactly that grad
    w_true=topk(cost,frac); w_spo=topk((2*sev-cost).detach(),frac)
    return (2*sev*(w_spo-w_true).detach()).mean()
def regret(sev_np,cost_np,frac=0.1):
    B=max(1,int(frac*len(cost_np)))
    return float(cost_np[np.argsort(-cost_np)[:B]].sum()-cost_np[np.argsort(-sev_np)[:B]].sum())
_BINS=np.arange(K,dtype=np.float32)
def risk_from_pmf(p): return -(p*_BINS).sum(1)
@torch.no_grad()
def predict(tr,mm,hd,sh,idx):
    tr.eval();mm.eval();hd.eval();sh.eval(); dlog,lam,clog,plog,sev=forward_all(tr,mm,hd,sh,idx)
    return (F.softmax(dlog,1).cpu().numpy(),lam.cpu().numpy(),torch.sigmoid(clog).cpu().numpy(),torch.sigmoid(plog).cpu().numpy(),sev.cpu().numpy())

In [ ]:
# 5 · fine-tune: anchor (full Tier-1 loss) + beta*SPO+ ; keep best-by-val-regret
params=list(trunk.parameters())+list(mmoe.parameters())+list(heads.parameters())+list(sev_head.parameters())
opt=torch.optim.Adam(params,lr=3e-4)
cost_va=(np.clip(va_t,None,CAP)*road[va_idx].cpu().numpy()*va_e)
EPOCHS,bs=30,1024
def beta(ep): return min(0.6, ep/10.0*0.6)        # gentle ramp, cap 0.6
best_r, best_state = 1e18, None
for ep in range(EPOCHS):
    trunk.train(); mmoe.train(); heads.train(); sev_head.train()
    perm=tr_idx[torch.randperm(tr_idx.numel(),device=device)]
    for s0 in range(0,perm.numel(),bs):
        b=perm[s0:s0+bs]
        if b.numel()<16: continue
        opt.zero_grad()
        dlog,lam,clog,plog,sev=forward_all(trunk,mmoe,heads,sev_head,b)
        loss=anchor_loss(dlog,lam,clog,plog,b)+beta(ep)*spo_plus_max(sev,cost_t[b])
        loss.backward(); torch.nn.utils.clip_grad_norm_(params,5.0); opt.step()
    sev_va=predict(trunk,mmoe,heads,sev_head,va_idx)[4]; r=regret(sev_va,cost_va)
    if r<best_r:
        best_r=r; best_state={'t':trunk.state_dict(),'m':mmoe.state_dict(),'h':heads.state_dict(),'s':sev_head.state_dict()}
    if ep%5==0: print('epoch %2d beta %.2f val_regret %.0f (best %.0f)'%(ep,beta(ep),r,best_r))
trunk.load_state_dict(best_state['t']); mmoe.load_state_dict(best_state['m']); heads.load_state_dict(best_state['h']); sev_head.load_state_dict(best_state['s'])
print('restored best val regret =', round(best_r,0))

In [ ]:
# 6 · evaluate: regret (SPO+ severity vs Tier-1 accuracy severity) + survival sanity
from sksurv.util import Surv
from sksurv.metrics import integrated_brier_score
te_t,te_e=durations[split==2],events[split==2]
cost_te=np.clip(te_t,None,CAP)*road[te_idx].cpu().numpy()*te_e
pmf_te,lam_te,clo_te,pri_te,sev_te=predict(trunk,mmoe,heads,sev_head,te_idx)
# Tier-1 'accuracy' severity = E[dur] x P(closure) from the untouched MTL model
b_tr,b_mm,b_hd=build(); load_mtl(b_tr,b_mm,b_hd)
with torch.no_grad():
    av=b_tr.event_vectors(X_num,X_cat); reps=b_mm(b_tr(te_idx,av)); dlog,_,clog,_=b_hd(reps)
    sev_te0=((F.softmax(dlog,1)*mids_t).sum(1)*torch.sigmoid(clog)).cpu().numpy()
B=max(1,int(0.1*len(cost_te))); oracle=cost_te[np.argsort(-cost_te)[:B]].sum()
print('=== decision regret @ budget 10%  (lower = better) ===')
print('  Tier-1 accuracy severity  regret %.0f'%regret(sev_te0,cost_te))
print('  Tier-1.5 SPO+ severity    regret %.0f'%regret(sev_te,cost_te))
print('  oracle captures %.0f of %.0f total severity-min'%(oracle,cost_te.sum()))
grid=np.linspace(np.percentile(te_t[te_e.astype(bool)],5),np.percentile(te_t[te_e.astype(bool)],95),20)
y_tr=Surv.from_arrays(events[split==0].astype(bool),np.clip(durations[split==0],1,None)); y_te=Surv.from_arrays(te_e.astype(bool),np.clip(te_t,1,None))
def surv_at(pmf,times):
    cum=np.cumsum(pmf,1); right=cuts[1:]; S=np.ones((pmf.shape[0],len(times)))
    for j,t in enumerate(times):
        k=np.searchsorted(right,t,side='right')
        if k>0: S[:,j]=1.0-cum[:,min(k-1,pmf.shape[1]-1)]
    return S
print('survival sanity (anchored): C-index %.4f  IBS %.4f  (target ~Tier-1: 0.586 / 0.051)'%(
    concordance_index_censored(te_e.astype(bool),te_t,risk_from_pmf(pmf_te))[0],
    integrated_brier_score(y_tr,y_te,surv_at(pmf_te,grid),grid)))

In [ ]:
# 7 · save SPO+ model + predictions
os.makedirs('out',exist_ok=True)
torch.save({'trunk':trunk.state_dict(),'mmoe':mmoe.state_dict(),'heads':heads.state_dict(),
            'sev_head':sev_head.state_dict(),'cuts':cuts,'config':ckpt['config']},'out/trunk_spo.pt')
pmf_all,lam_all,clo_all,pri_all,sev_all=predict(trunk,mmoe,heads,sev_head,torch.arange(N,device=device))
node_all=z['node_id']; node_intensity=np.zeros(V)
for v in range(V):
    m=node_all==v
    if m.any(): node_intensity[v]=float(lam_all[m].mean())
np.savez_compressed('out/preds_spo.npz',
    pmf=pmf_all, intensity=lam_all, node_intensity=node_intensity, severity=sev_all,
    closure_prob=clo_all, priority_prob=pri_all, cuts=cuts, event_id=z['event_id'], node_id=node_all,
    split=split, durations=durations, events=events, tte_min=z['tte_min'], tte_observed=z['tte_observed'],
    road_closure=z['road_closure'], priority_high=z['priority_high'])
print('saved out/trunk_spo.pt , out/preds_spo.npz')
try:
    from google.colab import files; files.download('out/trunk_spo.pt'); files.download('out/preds_spo.npz')
except Exception as e: print('(local)',e)

## Handoff

Place `trunk_spo.pt`, `preds_spo.npz` into local `models/` (overwrite the v1 ones).

Expect: **SPO+ regret < Tier-1 accuracy regret**, and **survival IBS back near ~0.05** (the anchor
protects calibration). Next local step points conformal + allocator at `preds_spo.npz` and uses its
decision-tuned `severity` so the plan captures more true congestion per unit budget.
